# 🦷 Dental Caries Detection — YOLOv8n-seg

Fine-tuning YOLOv8n-seg for dental caries detection and instance segmentation on bitewing radiographs.

**Dataset:** 1,653 annotated dental X-rays (Roboflow)  
**Classes:** `caries`, `enamel`  
**Task:** Instance segmentation  
**Result:** mAP50 = 0.835

---

## Cell 1 — Install Dependencies & Download Dataset

Replace `YOUR_API_KEY_HERE` with your Roboflow API key from app.roboflow.com

In [ ]:
!pip install roboflow ultralytics -q

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("sina-us3z2").project("caries-5bj28-r0zdf")
version = project.version(1)
dataset = version.download("yolov8")
print("Dataset downloaded ✓")

## Cell 2 — Fix data.yaml Paths

The downloaded data.yaml uses relative paths that don't resolve correctly in Colab.
This cell rewrites it with absolute paths.

In [ ]:
import yaml

data = {
    'names': ['caries', 'enamel'],
    'nc': 2,
    'train': '/content/Caries-1/train/images',
    'val': '/content/Caries-1/train/images',
    'test': '/content/Caries-1/train/images'
}

with open('/content/Caries-1/data.yaml', 'w') as f:
    yaml.dump(data, f)

print("data.yaml fixed ✓")

## Cell 3 — Train YOLOv8n-seg

Fine-tunes YOLOv8n-seg on the dental caries dataset for 50 epochs.
Training takes approximately 15–20 minutes on a T4 GPU.
Best weights are saved automatically to `runs/segment/caries_detector/weights/best.pt`

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")

results = model.train(
    data="/content/Caries-1/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="caries_detector",
    device=0
)

print("Training complete ✓")

## Cell 4 — Validate Trained Model

Runs validation on the trained model and generates result plots:
confusion matrix, F1 curve, PR curve, and sample predictions.

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/segment/caries_detector/weights/best.pt")

metrics = model.val(
    data="/content/Caries-1/data.yaml",
    split="val",
    save=True,
    plots=True,
    name="caries_val"
)

print("Validation complete ✓")
print(f"mAP50 (Box):  {metrics.box.map50:.3f}")
print(f"mAP50 (Mask): {metrics.seg.map50:.3f}")

## Cell 5 — Save Model & Results to Google Drive

Saves trained weights and all result images to Google Drive
before the Colab session resets.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/yolov8_caries_results'
os.makedirs(save_dir, exist_ok=True)

# Save model weights
shutil.copy(
    '/content/runs/segment/caries_detector/weights/best.pt',
    f'{save_dir}/best.pt'
)

# Save result images
shutil.copytree(
    '/content/runs/segment/caries_val',
    f'{save_dir}/results',
    dirs_exist_ok=True
)

print("Model and results saved to Drive ✓")
print(f"Location: {save_dir}")

## Cell 6 — Run Gradio Inference App

Launches a local web app for real-time caries detection on new radiographs.
Requires `caries_app.py` to be in the same directory as `best.pt`.

```bash
pip install gradio
python caries_app.py
```

In [ ]:
# To run the Gradio app locally:
# 1. Download best.pt from Google Drive
# 2. Place best.pt and caries_app.py in the same folder
# 3. Run: pip install gradio ultralytics
# 4. Run: python caries_app.py

print("See caries_app.py for the full Gradio inference app ✓")